# Data Validation and Quality Assessment

## Objective

This notebook validates the raw X5 RetailHero datasets before analytical processing. It checks file availability, file size, schema, row counts, key uniqueness, missing values, duplicate records, and treatment-control consistency.

Large transaction files are queried directly with DuckDB to avoid loading the full dataset into memory.

In [1]:
from pathlib import Path

import duckdb
import pandas as pd

# Identify the project root whether the notebook starts from the
# repository root or from the notebooks directory.
current_path = Path.cwd().resolve()

if (current_path / "data").exists():
    PROJECT_ROOT = current_path
elif (current_path.parent / "data").exists():
    PROJECT_ROOT = current_path.parent
else:
    raise FileNotFoundError("Project root could not be identified.")

RAW_DIR = PROJECT_ROOT / "data" / "raw"

expected_files = [
    "clients.csv",
    "products.csv",
    "purchases.csv",
    "uplift_train.csv",
    "uplift_test.csv",
    "uplift_sample_submission.csv",
]

file_inventory = []

for file_name in expected_files:
    file_path = RAW_DIR / file_name
    file_inventory.append(
        {
            "file_name": file_name,
            "exists": file_path.exists(),
            "size_mb": (
                round(file_path.stat().st_size / (1024**2), 2)
                if file_path.exists()
                else None
            ),
        }
    )

inventory_df = pd.DataFrame(file_inventory)

assert inventory_df["exists"].all(), "One or more expected data files are missing."

print(f"Project root: {PROJECT_ROOT}")
print(f"Raw data directory: {RAW_DIR}")

inventory_df

Project root: C:\Users\DaiXY\Documents\GitHub\retail-customer-uplift-analytics
Raw data directory: C:\Users\DaiXY\Documents\GitHub\retail-customer-uplift-analytics\data\raw


,file_name,exists,size_mb
0,clients.csv,True,20.73
1,products.csv,True,3.71
2,purchases.csv,True,4256.99
3,uplift_train.csv,True,2.86
4,uplift_test.csv,True,2.10
5,uplift_sample_submission.csv,True,5.78


## 1. Schema Inspection and Sample Preview

Inspect column names, inferred data types, and five sample rows from
each source file before defining analytical tables.

DuckDB uses a bounded sample for CSV schema inference. Only the schema
and five preview rows are returned as pandas DataFrames; the full
transaction dataset is not loaded into pandas.

Inferred types are provisional. Identifier types, date formats, and
numeric fields will be reviewed before full-data validation.

In [2]:
from IPython.display import display

schema_results = {}
sample_results = {}

with duckdb.connect(database=":memory:") as con:
    for file_name in expected_files:
        file_path = str(RAW_DIR / file_name)
        table_name = Path(file_name).stem

        csv_query = """
            SELECT *
            FROM read_csv_auto(
                ?,
                header = true,
                sample_size = 20480
            )
        """

        schema_df = con.execute(
            "DESCRIBE " + csv_query,
            [file_path]
        ).df()

        sample_df = con.execute(
            csv_query + " LIMIT 5",
            [file_path]
        ).df()

        schema_results[table_name] = schema_df
        sample_results[table_name] = sample_df

        print(f"\n{'=' * 60}")
        print(f"Table: {table_name} | Columns: {len(schema_df)}")

        display(schema_df[["column_name", "column_type"]])
        display(sample_df)


Table: clients | Columns: 5


,column_name,column_type
0,client_id,VARCHAR
1,first_issue_date,TIMESTAMP
2,first_redeem_date,TIMESTAMP
3,age,BIGINT
4,gender,VARCHAR


,client_id,first_issue_date,first_redeem_date,age,gender
0,000012768d,2017-08-05 15:40:48,2018-01-04 19:30:07,45,U
1,000036f903,2017-04-10 13:54:23,2017-04-23 12:37:56,72,F
2,000048b7a6,2018-12-15 13:33:11,NaT,68,F
3,000073194a,2017-05-23 12:56:14,2017-11-24 11:18:01,60,F
4,00007c7133,2017-05-22 16:17:08,2018-12-31 17:17:33,67,U



Table: products | Columns: 11


,column_name,column_type
0,product_id,VARCHAR
1,level_1,VARCHAR
2,level_2,VARCHAR
3,level_3,VARCHAR
4,level_4,VARCHAR
5,segment_id,DOUBLE
6,brand_id,VARCHAR
7,vendor_id,VARCHAR
8,netto,DOUBLE
9,is_own_trademark,BIGINT


...

,column_name,column_type
0,client_id,VARCHAR


,client_id
0,000048b7a6
1,000073194a
2,00007c7133
3,00007f9014
4,0000a90cf7



Table: uplift_sample_submission | Columns: 2


,column_name,column_type
0,client_id,VARCHAR
1,uplift,DOUBLE


,client_id,uplift
0,008fb49e3a,0.114991
1,0095340acc,0.835321
2,015c0b4d79,0.308584
3,017ec51c8e,0.415716
4,01bbc5ff71,0.034669


## 2. Row Count Validation

Count records in all six source files to establish dataset scale
and provide denominators for subsequent data quality checks.

Purchase rows represent transaction detail records and should not
be interpreted as distinct orders or customers.

DuckDB scans the CSV files directly and returns only aggregate counts.

In [3]:
row_count_results = []

with duckdb.connect(database=":memory:") as con:
    for file_name in expected_files:
        file_path = str(RAW_DIR / file_name)
        table_name = Path(file_name).stem

        print(f"Counting rows: {table_name}...", flush=True)

        row_count = con.execute(
            """
            SELECT COUNT(*) AS row_count
            FROM read_csv_auto(
                ?,
                header = true,
                sample_size = 20480
            )
            """,
            [file_path]
        ).fetchone()[0]

        row_count_results.append(
            {
                "table_name": table_name,
                "row_count": row_count,
                "column_count": len(schema_results[table_name]),
            }
        )

row_counts_df = pd.DataFrame(row_count_results)

display(row_counts_df)

Counting rows: clients...
Counting rows: products...
Counting rows: purchases...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Counting rows: uplift_train...
Counting rows: uplift_test...
Counting rows: uplift_sample_submission...


,table_name,row_count,column_count
0,clients,400162,5
1,products,43038,11
2,purchases,45786568,13
3,uplift_train,200039,3
4,uplift_test,200123,1
5,uplift_sample_submission,200123,2


## 3. Key Uniqueness Validation

Check candidate keys in the customer, product, and experiment files.

A valid key must be non-null, non-blank, and unique. Duplicate keys
can multiply records during joins and distort business metrics.

Repeated transaction IDs in purchases are expected when an order
contains multiple product detail rows. Purchase duplicates will be
assessed separately.

In [4]:
key_columns = {
    "clients": "client_id",
    "products": "product_id",
    "uplift_train": "client_id",
    "uplift_test": "client_id",
    "uplift_sample_submission": "client_id",
}

key_check_results = []

with duckdb.connect(database=":memory:") as con:
    for table_name, key_column in key_columns.items():
        file_path = str(RAW_DIR / f"{table_name}.csv")

        result = con.execute(
            f"""
            SELECT
                COUNT(*) AS row_count,
                COUNT(*) FILTER (
                    WHERE {key_column} IS NULL
                ) AS null_key_count,
                COUNT(*) FILTER (
                    WHERE TRIM({key_column}) = ''
                ) AS blank_key_count,
                COUNT(DISTINCT {key_column}) AS distinct_key_count,
                COUNT({key_column})
                    - COUNT(DISTINCT {key_column})
                    AS duplicate_excess_rows
            FROM read_csv_auto(
                ?,
                header = true,
                sample_size = 20480
            )
            """,
            [file_path]
        ).df()

        result.insert(0, "table_name", table_name)
        result.insert(1, "key_column", key_column)
        key_check_results.append(result)

key_checks_df = pd.concat(key_check_results, ignore_index=True)

key_checks_df["key_valid"] = (
    (key_checks_df["null_key_count"] == 0)
    & (key_checks_df["blank_key_count"] == 0)
    & (key_checks_df["duplicate_excess_rows"] == 0)
)

display(key_checks_df)

,table_name,key_column,row_count,null_key_count,blank_key_count,distinct_key_count,duplicate_excess_rows,key_valid
0,clients,client_id,400162,0,0,400162,0,True
1,products,product_id,43038,0,0,43038,0,True
2,uplift_train,client_id,200039,0,0,200039,0,True
3,uplift_test,client_id,200123,0,0,200123,0,True
4,uplift_sample_submission,client_id,200123,0,0,200123,0,True


## 3.1 Purchase Detail Duplicate Check

Identify purchase records whose values match across all 13 parsed
columns. Repeated transaction IDs alone are not treated as duplicates.

Report the number of repeated groups and excess rows. No records
are removed at this stage because identical records may require
further investigation of the source data and transaction grain.

DuckDB uses a memory limit and a local temporary directory for
intermediate processing.

In [5]:
temp_dir = PROJECT_ROOT / "data" / "processed" / "duckdb_temp"
temp_dir.mkdir(parents=True, exist_ok=True)

purchase_path = str(RAW_DIR / "purchases.csv")

with duckdb.connect(database=":memory:") as con:
    con.execute("SET memory_limit = '2GB'")
    con.execute("SET threads = 2")
    con.execute("SET preserve_insertion_order = false")
    con.execute(
        "SET temp_directory = ?",
        [str(temp_dir)]
    )

    print("Checking purchase detail duplicates...", flush=True)

    purchase_duplicate_df = con.execute(
        """
        WITH repeated_groups AS (
            SELECT
                *,
                COUNT(*) AS occurrence_count
            FROM read_csv_auto(
                ?,
                header = true,
                sample_size = 20480
            )
            GROUP BY ALL
            HAVING COUNT(*) > 1
        )
        SELECT
            COUNT(*) AS duplicate_group_count,
            COALESCE(SUM(occurrence_count), 0)
                AS rows_in_duplicate_groups,
            COALESCE(SUM(occurrence_count - 1), 0)
                AS duplicate_excess_rows
        FROM repeated_groups
        """,
        [purchase_path]
    ).df()

display(purchase_duplicate_df)

Checking purchase detail duplicates...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,duplicate_group_count,rows_in_duplicate_groups,duplicate_excess_rows
0,0,0.0,0.0


## 4. Missing Value Assessment

Measure null counts and null percentages for every column in all
six source files. Separately count blank or whitespace-only strings
in VARCHAR columns.

Missing values are reported without imputation or record removal.
Their meaning will be reviewed before defining cleaning rules.

Results reflect DuckDB's CSV parsing and default null handling.

In [6]:
missing_results = []

with duckdb.connect(database=":memory:") as con:
    for file_name in expected_files:
        table_name = Path(file_name).stem
        file_path = str(RAW_DIR / file_name)
        schema_df = schema_results[table_name]

        expressions = ["COUNT(*) AS total_rows"]

        for i, row in schema_df.iterrows():
            column_name = row["column_name"]
            column_type = row["column_type"]
            quoted_column = '"' + column_name.replace('"', '""') + '"'

            expressions.append(
                f"COUNT(*) FILTER (WHERE {quoted_column} IS NULL) "
                f"AS null_{i}"
            )

            if column_type == "VARCHAR":
                expressions.append(
                    f"COUNT(*) FILTER "
                    f"(WHERE TRIM({quoted_column}) = '') "
                    f"AS blank_{i}"
                )

        query = f"""
            SELECT {", ".join(expressions)}
            FROM read_csv_auto(
                ?,
                header = true,
                sample_size = 20480
            )
        """

        print(f"Checking missing values: {table_name}...", flush=True)
        counts = con.execute(query, [file_path]).df().iloc[0]
        total_rows = int(counts["total_rows"])

        for i, row in schema_df.iterrows():
            null_count = int(counts[f"null_{i}"])
            blank_count = (
                int(counts[f"blank_{i}"])
                if row["column_type"] == "VARCHAR"
                else None
            )

            missing_results.append(
                {
                    "table_name": table_name,
                    "column_name": row["column_name"],
                    "column_type": row["column_type"],
                    "total_rows": total_rows,
                    "null_count": null_count,
                    "null_pct": (
                        round(100 * null_count / total_rows, 4)
                        if total_rows > 0
                        else None
                    ),
                    "blank_string_count": blank_count,
                }
            )

missing_values_df = pd.DataFrame(missing_results)

for table_name in missing_values_df["table_name"].unique():
    print(f"\nTable: {table_name}")
    display(
        missing_values_df.loc[
            missing_values_df["table_name"] == table_name
        ].reset_index(drop=True)
    )

Checking missing values: clients...
Checking missing values: products...
Checking missing values: purchases...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Checking missing values: uplift_train...
Checking missing values: uplift_test...
Checking missing values: uplift_sample_submission...

Table: clients


,table_name,column_name,column_type,total_rows,null_count,null_pct,blank_string_count
0,clients,client_id,VARCHAR,400162,0,0.0000,0.0
1,clients,first_issue_date,TIMESTAMP,400162,0,0.0000,NaN
2,clients,first_redeem_date,TIMESTAMP,400162,35469,8.8637,NaN
3,clients,age,BIGINT,400162,0,0.0000,NaN
4,clients,gender,VARCHAR,400162,0,0.0000,0.0



Table: products


...

,table_name,column_name,column_type,total_rows,null_count,null_pct,blank_string_count
0,uplift_train,client_id,VARCHAR,200039,0,0.0,0.0
1,uplift_train,treatment_flg,BIGINT,200039,0,0.0,NaN
2,uplift_train,target,BIGINT,200039,0,0.0,NaN



Table: uplift_test


,table_name,column_name,column_type,total_rows,null_count,null_pct,blank_string_count
0,uplift_test,client_id,VARCHAR,200123,0,0.0,0.0



Table: uplift_sample_submission


,table_name,column_name,column_type,total_rows,null_count,null_pct,blank_string_count
0,uplift_sample_submission,client_id,VARCHAR,200123,0,0.0,0.0
1,uplift_sample_submission,uplift,DOUBLE,200123,0,0.0,NaN


## 4.1 Customer and Product Value Checks

Inspect age ranges, gender codes, and product flag values to identify
non-null values that may require review.

Ages below 0 or above 100 are flagged for investigation, not
automatically removed. Product flags are checked against {0, 1}.
No cleaning decisions are applied at this stage.

In [7]:
clients_path = str(RAW_DIR / "clients.csv")
products_path = str(RAW_DIR / "products.csv")

with duckdb.connect(database=":memory:") as con:
    age_check_df = con.execute(
        """
        SELECT
            COUNT(*) AS customer_count,
            MIN(age) AS min_age,
            MAX(age) AS max_age,
            COUNT(*) FILTER (WHERE age < 0) AS negative_age_count,
            COUNT(*) FILTER (WHERE age = 0) AS zero_age_count,
            COUNT(*) FILTER (WHERE age > 100) AS age_over_100_count
        FROM read_csv_auto(
            ?, header = true, sample_size = 20480
        )
        """,
        [clients_path]
    ).df()

    gender_counts_df = con.execute(
        """
        SELECT
            gender,
            COUNT(*) AS customer_count,
            ROUND(
                100.0 * COUNT(*) / SUM(COUNT(*)) OVER (),
                4
            ) AS customer_pct
        FROM read_csv_auto(
            ?, header = true, sample_size = 20480
        )
        GROUP BY gender
        ORDER BY customer_count DESC, gender
        """,
        [clients_path]
    ).df()

    product_flag_results = []

    for flag_column in ["is_own_trademark", "is_alcohol"]:
        result = con.execute(
            f"""
            SELECT
                {flag_column} AS flag_value,
                COUNT(*) AS product_count,
                CASE
                    WHEN {flag_column} IN (0, 1) THEN true
                    ELSE false
                END AS value_valid
            FROM read_csv_auto(
                ?, header = true, sample_size = 20480
            )
            GROUP BY {flag_column}
            ORDER BY flag_value
            """,
            [products_path]
        ).df()

        result.insert(0, "flag_column", flag_column)
        product_flag_results.append(result)

product_flags_df = pd.concat(
    product_flag_results,
    ignore_index=True
)

print("Age checks")
display(age_check_df)

print("Gender distribution")
display(gender_counts_df)

print("Product flag checks")
display(product_flags_df)

Age checks


,customer_count,min_age,max_age,negative_age_count,zero_age_count,age_over_100_count
0,400162,-7491,1901,96,42,1049


Gender distribution


,gender,customer_count,customer_pct
0,U,185706,46.4077
1,F,147649,36.8973
2,M,66807,16.6950


Product flag checks


,flag_column,flag_value,product_count,value_valid
0,is_own_trademark,0,41524,True
1,is_own_trademark,1,1514,True
2,is_alcohol,0,40645,True
3,is_alcohol,1,2393,True


## 4.2 Flagged Age Distribution

Inspect the frequency of each age value flagged for review:
negative ages, zero ages, and ages above 100.

This check distinguishes clearly invalid values from potentially
valid extreme ages before a cleaning rule is defined.
All source records remain unchanged.

In [8]:
with duckdb.connect(database=":memory:") as con:
    flagged_age_distribution_df = con.execute(
        """
        SELECT
            age,
            COUNT(*) AS customer_count,
            CASE
                WHEN age < 0 THEN 'Negative age'
                WHEN age = 0 THEN 'Zero age'
                ELSE 'Age above 100'
            END AS review_reason
        FROM read_csv_auto(
            ?,
            header = true,
            sample_size = 20480
        )
        WHERE age <= 0 OR age > 100
        GROUP BY age
        ORDER BY age
        """,
        [clients_path]
    ).df()

print(
    "Customers flagged for age review:",
    int(flagged_age_distribution_df["customer_count"].sum())
)

print(
    "Distinct flagged age values:",
    len(flagged_age_distribution_df)
)

display(flagged_age_distribution_df)

Customers flagged for age review: 1187
Distinct flagged age values: 191


,age,customer_count,review_reason
0,-7491,1,Negative age
1,-7158,1,Negative age
2,-5973,1,Negative age
3,-5953,1,Negative age
4,-5949,1,Negative age
...,...,...,...
186,1717,1,Age above 100
187,1800,1,Age above 100
188,1841,1,Age above 100
189,1852,1,Age above 100


### Review of Flagged Age Values

Inspect ages from 101 to 120 and the most frequent flagged values.
These views support a documented age-cleaning decision.
The range 101–120 is a review window, not a validated age range.

In [9]:
high_age_review_df = flagged_age_distribution_df.loc[
    flagged_age_distribution_df["age"].between(101, 120)
].copy()

frequent_flagged_ages_df = (
    flagged_age_distribution_df
    .sort_values(
        ["customer_count", "age"],
        ascending=[False, True]
    )
    .head(20)
)

print("Age values from 101 to 120")
display(high_age_review_df)

print("20 most frequent flagged age values")
display(frequent_flagged_ages_df)

Age values from 101 to 120


,age,customer_count,review_reason
65,101,1,Age above 100
66,102,3,Age above 100
67,105,2,Age above 100
68,106,1,Age above 100
69,108,4,Age above 100
70,109,5,Age above 100
71,110,1,Age above 100
72,111,1,Age above 100
73,114,5,Age above 100
74,115,405,Age above 100


20 most frequent flagged age values


,age,customer_count,review_reason
74,115,405,Age above 100
78,119,397,Age above 100
64,0,42,Zero age
149,944,7,Age above 100
153,949,7,Age above 100
141,934,6,Age above 100
150,946,6,Age above 100
156,952,6,Age above 100
43,-943,5,Negative age
70,109,5,Age above 100


### Provisional Age Handling Rule

Age values of 115 and 119 occur for 405 and 397 customers,
respectively. Their concentration suggests a possible data quality
issue, but the underlying cause cannot be established.

For the primary analysis:
- Preserve the original age as age_raw.
- Define age_clean using ages from 1 to 100; set other values to null.
- Flag ages outside this range using age_requires_review.
- Retain all customer, transaction, and experiment records.
- Report affected customers as age unknown / under review.

This range is an analytical convention, not proof of age validity.
The rule will be implemented in the customer data layer.

## 5.1 Experiment Customer Relationship Checks

Check whether train and test customer sets overlap, whether all
experiment customers exist in clients, and whether their union
covers the customer table.

Matching row counts alone do not establish matching customer sets.

In [10]:
with duckdb.connect(database=":memory:") as con:
    for table_name in ["clients", "uplift_train", "uplift_test"]:
        file_path = str(RAW_DIR / f"{table_name}.csv")

        con.execute(
            f"""
            CREATE TEMP TABLE {table_name} AS
            SELECT client_id
            FROM read_csv_auto(
                ?, header = true, sample_size = 20480
            )
            """,
            [file_path]
        )

    experiment_relationship_df = con.execute(
        """
        WITH experiment_ids AS (
            SELECT client_id FROM uplift_train
            UNION
            SELECT client_id FROM uplift_test
        )
        SELECT
            'train_test_overlap' AS check_name,
            COUNT(*) AS issue_count
        FROM uplift_train AS t
        INNER JOIN uplift_test AS s
            ON t.client_id = s.client_id

        UNION ALL

        SELECT
            'train_clients_missing_from_clients',
            COUNT(*)
        FROM uplift_train AS t
        WHERE NOT EXISTS (
            SELECT 1
            FROM clients AS c
            WHERE c.client_id = t.client_id
        )

        UNION ALL

        SELECT
            'test_clients_missing_from_clients',
            COUNT(*)
        FROM uplift_test AS t
        WHERE NOT EXISTS (
            SELECT 1
            FROM clients AS c
            WHERE c.client_id = t.client_id
        )

        UNION ALL

        SELECT
            'clients_outside_train_and_test',
            COUNT(*)
        FROM clients AS c
        WHERE NOT EXISTS (
            SELECT 1
            FROM experiment_ids AS e
            WHERE e.client_id = c.client_id
        )
        """
    ).df()

display(experiment_relationship_df)

,check_name,issue_count
0,train_test_overlap,0
1,train_clients_missing_from_clients,0
2,test_clients_missing_from_clients,0
3,clients_outside_train_and_test,0


## 5.2 Purchase Foreign Key Checks

Check whether every purchase detail row has a matching customer
in clients and a matching product in products.

Unmatched records can cause data loss during inner joins or missing
attributes during left joins. Report affected detail rows without
removing any records.

In [11]:
with duckdb.connect(database=":memory:") as con:
    for table_name, key_column in [
        ("clients", "client_id"),
        ("products", "product_id"),
    ]:
        file_path = str(RAW_DIR / f"{table_name}.csv")

        con.execute(
            f"""
            CREATE TEMP TABLE {table_name} AS
            SELECT {key_column}
            FROM read_csv_auto(
                ?, header = true, sample_size = 20480
            )
            """,
            [file_path]
        )

    print("Checking purchase customer and product matches...", flush=True)

    purchase_relationship_df = con.execute(
        """
        SELECT
            COUNT(*) AS purchase_detail_rows,
            COUNT(*) FILTER (
                WHERE c.client_id IS NULL
            ) AS unmatched_customer_rows,
            COUNT(*) FILTER (
                WHERE pr.product_id IS NULL
            ) AS unmatched_product_rows
        FROM read_csv_auto(
            ?, header = true, sample_size = 20480
        ) AS p
        LEFT JOIN clients AS c
            ON p.client_id = c.client_id
        LEFT JOIN products AS pr
            ON p.product_id = pr.product_id
        """,
        [str(RAW_DIR / "purchases.csv")]
    ).df()

display(purchase_relationship_df)

Checking purchase customer and product matches...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,purchase_detail_rows,unmatched_customer_rows,unmatched_product_rows
0,45786568,0,0


## 5.3 Submission Customer Coverage

Verify that uplift_sample_submission and uplift_test contain exactly
the same customer IDs. Row order is not required to match.

The example uplift values are not observed outcomes and are not
used for model evaluation.

In [12]:
with duckdb.connect(database=":memory:") as con:
    for table_name in ["uplift_test", "uplift_sample_submission"]:
        file_path = str(RAW_DIR / f"{table_name}.csv")

        con.execute(
            f"""
            CREATE TEMP TABLE {table_name} AS
            SELECT client_id
            FROM read_csv_auto(
                ?, header = true, sample_size = 20480
            )
            """,
            [file_path]
        )

    submission_coverage_df = con.execute(
        """
        SELECT
            'test_ids_missing_from_submission' AS check_name,
            COUNT(*) AS issue_count
        FROM uplift_test AS t
        WHERE NOT EXISTS (
            SELECT 1
            FROM uplift_sample_submission AS s
            WHERE s.client_id = t.client_id
        )

        UNION ALL

        SELECT
            'submission_ids_missing_from_test',
            COUNT(*)
        FROM uplift_sample_submission AS s
        WHERE NOT EXISTS (
            SELECT 1
            FROM uplift_test AS t
            WHERE t.client_id = s.client_id
        )
        """
    ).df()

display(submission_coverage_df)

,check_name,issue_count
0,test_ids_missing_from_submission,0
1,submission_ids_missing_from_test,0


## 6. Experiment Field Validation

Validate that treatment_flg and target contain only binary values
{0, 1}. Inspect treatment-control sample sizes and the joint
distribution of treatment assignment and purchase outcomes.

These checks establish coding validity and observed group sizes.
They do not prove randomization or campaign effectiveness.

In [13]:
train_path = str(RAW_DIR / "uplift_train.csv")

with duckdb.connect(database=":memory:") as con:
    con.execute(
        """
        CREATE TEMP TABLE experiment AS
        SELECT client_id, treatment_flg, target
        FROM read_csv_auto(
            ?, header = true, sample_size = 20480
        )
        """,
        [train_path]
    )

    experiment_validity_df = con.execute(
        """
        SELECT
            COUNT(*) AS customer_count,
            COUNT(*) FILTER (
                WHERE treatment_flg IS NULL
                   OR treatment_flg NOT IN (0, 1)
            ) AS invalid_treatment_count,
            COUNT(*) FILTER (
                WHERE target IS NULL
                   OR target NOT IN (0, 1)
            ) AS invalid_target_count
        FROM experiment
        """
    ).df()

    treatment_counts_df = con.execute(
        """
        SELECT
            treatment_flg,
            COUNT(*) AS customer_count,
            ROUND(
                100.0 * COUNT(*) / SUM(COUNT(*)) OVER (),
                4
            ) AS customer_pct
        FROM experiment
        GROUP BY treatment_flg
        ORDER BY treatment_flg
        """
    ).df()

    experiment_combinations_df = con.execute(
        """
        SELECT
            treatment_flg,
            target,
            COUNT(*) AS customer_count
        FROM experiment
        GROUP BY treatment_flg, target
        ORDER BY treatment_flg, target
        """
    ).df()

print("Binary field validity")
display(experiment_validity_df)

print("Treatment-control sample sizes")
display(treatment_counts_df)

print("Treatment and outcome combinations")
display(experiment_combinations_df)

Binary field validity


,customer_count,invalid_treatment_count,invalid_target_count
0,200039,0,0


Treatment-control sample sizes


,treatment_flg,customer_count,customer_pct
0,0,100058,50.0192
1,1,99981,49.9808


Treatment and outcome combinations


,treatment_flg,target,customer_count
0,0,0,39695
1,0,1,60363
2,1,0,36342
3,1,1,63639


## 7. Date Range and Sequence Checks

Inspect the observed ranges of card issue dates, first redemption
dates, and transaction timestamps.

Flag first redemption dates earlier than card issue dates.
Missing redemption dates are reported separately and are not
treated as sequence violations.

Observed date ranges alone do not establish the campaign date
or prove that every feature was available before treatment.

In [14]:
with duckdb.connect(database=":memory:") as con:
    client_date_checks_df = con.execute(
        """
        SELECT
            MIN(first_issue_date) AS earliest_issue_date,
            MAX(first_issue_date) AS latest_issue_date,
            MIN(first_redeem_date) AS earliest_redeem_date,
            MAX(first_redeem_date) AS latest_redeem_date,
            COUNT(*) FILTER (
                WHERE first_redeem_date IS NULL
            ) AS missing_redeem_date_count,
            COUNT(*) FILTER (
                WHERE first_redeem_date < first_issue_date
            ) AS redeem_before_issue_count
        FROM read_csv_auto(
            ?, header = true, sample_size = 20480
        )
        """,
        [str(RAW_DIR / "clients.csv")]
    ).df()

    print("Checking transaction date range...", flush=True)

    transaction_date_checks_df = con.execute(
        """
        SELECT
            MIN(transaction_datetime) AS earliest_transaction,
            MAX(transaction_datetime) AS latest_transaction,
            COUNT(*) FILTER (
                WHERE transaction_datetime IS NULL
            ) AS missing_transaction_date_count
        FROM read_csv_auto(
            ?, header = true, sample_size = 20480
        )
        """,
        [str(RAW_DIR / "purchases.csv")]
    ).df()

print("Customer date checks")
display(client_date_checks_df.T.rename(columns={0: "value"}))

print("Transaction date checks")
display(transaction_date_checks_df.T.rename(columns={0: "value"}))

Checking transaction date range...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Customer date checks


,value
earliest_issue_date,2017-04-04 18:24:18
latest_issue_date,2019-03-15 21:50:56
earliest_redeem_date,2017-04-11 09:42:20
latest_redeem_date,2019-11-20 01:14:10
missing_redeem_date_count,35469
redeem_before_issue_count,536


Transaction date checks


,value
earliest_transaction,2018-11-21 21:02:33
latest_transaction,2019-03-18 23:40:03
missing_transaction_date_count,0


## 7.1 Loyalty Date Availability Review

Use the maximum observed transaction timestamp as a provisional
historical feature cutoff, not as the confirmed campaign start.

Count loyalty dates after this cutoff and distinguish missing dates,
dates before card issue, and dates within the historical window.

Post-cutoff redemption dates must not be used directly as historical
predictors. Feature eligibility requires further timing validation.

In [15]:
feature_cutoff = pd.Timestamp(
    transaction_date_checks_df.loc[0, "latest_transaction"]
).to_pydatetime()

print("Provisional historical feature cutoff:", feature_cutoff)

with duckdb.connect(database=":memory:") as con:
    loyalty_date_review_df = con.execute(
        """
        WITH date_status AS (
            SELECT
                first_issue_date,
                CASE
                    WHEN first_redeem_date IS NULL
                        THEN 'Missing redemption date'
                    WHEN first_redeem_date < first_issue_date
                        THEN 'Redemption before card issue'
                    WHEN first_redeem_date > ?
                        THEN 'Redemption after cutoff'
                    ELSE 'Redemption within historical window'
                END AS redemption_status
            FROM read_csv_auto(
                ?, header = true, sample_size = 20480
            )
        )
        SELECT
            redemption_status,
            COUNT(*) AS customer_count,
            COUNT(*) FILTER (
                WHERE first_issue_date > ?
            ) AS issue_after_cutoff_count
        FROM date_status
        GROUP BY redemption_status
        ORDER BY redemption_status
        """,
        [
            feature_cutoff,
            str(RAW_DIR / "clients.csv"),
            feature_cutoff,
        ]
    ).df()

display(loyalty_date_review_df)

Provisional historical feature cutoff: 2019-03-18 23:40:03


,redemption_status,customer_count,issue_after_cutoff_count
0,Missing redemption date,35469,0
1,Redemption after cutoff,44953,0
2,Redemption before card issue,536,0
3,Redemption within historical window,319204,0


### Loyalty Date Feature Policy

The provisional historical cutoff is 2019-03-18 23:40:03,
the maximum observed transaction timestamp.

There are 44,953 customers with a first redemption date after this
cutoff and 536 customers whose first redemption precedes card issue.

Preserve all raw fields and customer records. Exclude first_redeem_date
and its derived features from the primary model until their historical
availability can be established.

Do not use post-cutoff redemption status or date-quality categories
as model predictors. These categories are for data quality review only.

The cutoff is not a confirmed campaign start date.

## 7.2 Transaction Numeric Value Checks

Inspect the minimum, maximum, negative count, and zero count for
transaction amount, quantity, and point fields.

Negative or zero values are reported for interpretation before
cleaning. Purchase amounts and points may repeat across product
detail rows and are not summed in this check.

In [16]:
numeric_columns = [
    "purchase_sum",
    "product_quantity",
    "regular_points_received",
    "express_points_received",
    "regular_points_spent",
    "express_points_spent",
    "trn_sum_from_iss",
    "trn_sum_from_red",
]

expressions = []

for i, column in enumerate(numeric_columns):
    expressions.extend([
        f"MIN({column}) AS min_{i}",
        f"MAX({column}) AS max_{i}",
        f"COUNT(*) FILTER (WHERE {column} < 0) AS negative_{i}",
        f"COUNT(*) FILTER (WHERE {column} = 0) AS zero_{i}",
    ])

with duckdb.connect(database=":memory:") as con:
    print("Checking transaction numeric values...", flush=True)

    numeric_stats = con.execute(
        f"""
        SELECT {", ".join(expressions)}
        FROM read_csv_auto(
            ?, header = true, sample_size = 20480
        )
        """,
        [str(RAW_DIR / "purchases.csv")]
    ).df().iloc[0]

numeric_check_results = []

for i, column in enumerate(numeric_columns):
    numeric_check_results.append({
        "column_name": column,
        "min_value": numeric_stats[f"min_{i}"],
        "max_value": numeric_stats[f"max_{i}"],
        "negative_row_count": int(numeric_stats[f"negative_{i}"]),
        "zero_row_count": int(numeric_stats[f"zero_{i}"]),
    })

transaction_numeric_checks_df = pd.DataFrame(numeric_check_results)

display(transaction_numeric_checks_df)

Checking transaction numeric values...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,column_name,min_value,max_value,negative_row_count,zero_row_count
0,purchase_sum,0.0,35149.04,0,620
1,product_quantity,0.0,14941.00,0,2707278
2,regular_points_received,0.0,2399.00,0,280212
3,express_points_received,0.0,300.00,0,45729438
4,regular_points_spent,-5066.0,0.00,3004108,42782460
5,express_points_spent,-300.0,0.00,504616,45281952
6,trn_sum_from_iss,0.0,35149.00,0,713255
7,trn_sum_from_red,0.0,8789.00,0,34637


## 7.3 Transaction Grain and Amount Consistency

Test whether each transaction_id maps to one customer, store,
timestamp, and purchase_sum value.

This check supports the design of an order-level table and prevents
repeated order amounts from being summed across product detail rows.

A consistent value within a transaction does not, by itself,
establish the business definition of the amount field.

In [17]:
temp_dir = PROJECT_ROOT / "data" / "processed" / "duckdb_temp"
temp_dir.mkdir(parents=True, exist_ok=True)

with duckdb.connect(database=":memory:") as con:
    con.execute("SET memory_limit = '2GB'")
    con.execute("SET threads = 2")
    con.execute("SET preserve_insertion_order = false")
    con.execute("SET temp_directory = ?", [str(temp_dir)])

    print("Checking transaction grain and amount consistency...", flush=True)

    transaction_grain_checks_df = con.execute(
        """
        WITH transaction_checks AS (
            SELECT
                transaction_id,
                COUNT(*) AS detail_rows,
                COUNT(DISTINCT client_id) AS client_count,
                COUNT(DISTINCT store_id) AS store_count,
                COUNT(DISTINCT transaction_datetime) AS timestamp_count,
                COUNT(DISTINCT purchase_sum) AS amount_count
            FROM read_csv_auto(
                ?, header = true, sample_size = 20480
            )
            GROUP BY transaction_id
        )
        SELECT
            COUNT(*) AS distinct_transaction_ids,
            SUM(detail_rows) AS purchase_detail_rows,
            COUNT(*) FILTER (
                WHERE detail_rows > 1
            ) AS transactions_with_multiple_rows,
            COUNT(*) FILTER (
                WHERE client_count > 1
            ) AS ids_with_multiple_clients,
            COUNT(*) FILTER (
                WHERE store_count > 1
            ) AS ids_with_multiple_stores,
            COUNT(*) FILTER (
                WHERE timestamp_count > 1
            ) AS ids_with_multiple_timestamps,
            COUNT(*) FILTER (
                WHERE amount_count > 1
            ) AS ids_with_multiple_amounts
        FROM transaction_checks
        """,
        [str(RAW_DIR / "purchases.csv")]
    ).df()

display(transaction_grain_checks_df.T.rename(columns={0: "value"}))

Checking transaction grain and amount consistency...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,value
distinct_transaction_ids,8045201.0
purchase_detail_rows,45786568.0
transactions_with_multiple_rows,6977737.0
ids_with_multiple_clients,28.0
ids_with_multiple_stores,28.0
ids_with_multiple_timestamps,28.0
ids_with_multiple_amounts,28.0


## 7.4 Candidate Order Key Validation

Some transaction IDs map to multiple customers, stores, timestamps,
or amounts. Therefore, transaction_id alone is not a validated
global order key.

Test the candidate composite key:
client_id, transaction_id, store_id, transaction_datetime.

Purchase_sum is excluded from the key so that amount inconsistencies
remain detectable.

In [18]:
with duckdb.connect(database=":memory:") as con:
    con.execute("SET memory_limit = '2GB'")
    con.execute("SET threads = 2")
    con.execute("SET preserve_insertion_order = false")
    con.execute("SET temp_directory = ?", [str(temp_dir)])

    print("Checking candidate composite order key...", flush=True)

    composite_order_checks_df = con.execute(
        """
        WITH candidate_orders AS (
            SELECT
                client_id,
                transaction_id,
                store_id,
                transaction_datetime,
                COUNT(*) AS detail_rows,
                COUNT(DISTINCT purchase_sum) AS amount_count
            FROM read_csv_auto(
                ?, header = true, sample_size = 20480
            )
            GROUP BY
                client_id,
                transaction_id,
                store_id,
                transaction_datetime
        )
        SELECT
            COUNT(*) AS candidate_order_count,
            SUM(detail_rows) AS purchase_detail_rows,
            COUNT(*) FILTER (
                WHERE amount_count <> 1
            ) AS orders_without_one_amount,
            MAX(detail_rows) AS max_detail_rows_per_order
        FROM candidate_orders
        """,
        [str(RAW_DIR / "purchases.csv")]
    ).df()

display(composite_order_checks_df.T.rename(columns={0: "value"}))

Checking candidate composite order key...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,value
candidate_order_count,8045229.0
purchase_detail_rows,45786568.0
orders_without_one_amount,0.0
max_detail_rows_per_order,116.0


### Provisional Order Grain and Amount Rule

The candidate order key is:
client_id, transaction_id, store_id, transaction_datetime.

This key produces 8,045,229 candidate orders from 45,786,568 detail
rows. Every candidate order has exactly one distinct purchase_sum.
Using transaction_id alone produces 8,045,201 groups and merges
transactions with conflicting attributes.

For the order-level mart:
- Group by all four key fields.
- Retain purchase_sum once per order using MIN or MAX after
  validating within-order consistency.
- Do not sum purchase_sum across product detail rows.
- Preserve all source detail rows.
- Validate point fields separately before order-level aggregation.

The amount's business definition and currency remain unconfirmed.
The composite key is an analytical order definition based on the
available fields, not a source-system guarantee.

## 8. Data Quality Summary

Summarize validated checks, identified issues, and planned handling
rules for each source table.

Passing structural checks does not imply that all business definitions
or causal assumptions have been verified. Open questions are retained
for follow-up before downstream analysis.

In [19]:
quality_notes = {
    "clients": {
        "candidate_key": "client_id",
        "validated_checks": (
            "Unique non-null key; complete train/test coverage"
        ),
        "main_issues": (
            "1,187 ages flagged; 35,469 missing redemption dates; "
            "536 redemption-before-issue cases; "
            "44,953 post-cutoff redemption dates"
        ),
        "planned_action": (
            "Retain customers; preserve raw age; apply provisional "
            "age rule; exclude redemption-date features from primary model"
        ),
    },
    "products": {
        "candidate_key": "product_id",
        "validated_checks": (
            "Unique non-null key; binary product flags valid"
        ),
        "main_issues": (
            "Missing hierarchy, segment, brand, vendor and netto values; "
            "segment_id inferred as DOUBLE"
        ),
        "planned_action": (
            "Retain products; preserve unknown categories; "
            "review segment coding and attribute definitions"
        ),
    },
    "purchases": {
        "candidate_key": (
            "Order: client_id + transaction_id + store_id "
            "+ transaction_datetime; detail key unconfirmed"
        ),
        "validated_checks": (
            "No identical parsed rows; complete customer/product matches; "
            "8,045,229 candidate orders with consistent purchase_sum"
        ),
        "main_issues": (
            "28 transaction IDs have multiple clients; "
            "2,707,278 zero-quantity rows; "
            "93.3532% missing trn_sum_from_red; negative points spent"
        ),
        "planned_action": (
            "Use composite order key; retain amount once per order; "
            "review quantities, point aggregation and amount definitions"
        ),
    },
    "uplift_train": {
        "candidate_key": "client_id",
        "validated_checks": (
            "Unique non-null key; binary fields valid; "
            "100,058 control and 99,981 treatment customers"
        ),
        "main_issues": (
            "Assignment mechanism, expected allocation ratio, "
            "campaign timing and outcome window require source confirmation"
        ),
        "planned_action": (
            "Confirm experiment design; assess SRM where justified "
            "and pre-treatment covariate balance before effect reporting"
        ),
    },
    "uplift_test": {
        "candidate_key": "client_id",
        "validated_checks": (
            "Unique non-null key; no train overlap; "
            "all IDs present in clients"
        ),
        "main_issues": "No public treatment assignment or outcome",
        "planned_action": (
            "Use for optional scoring; evaluate models on held-out "
            "customers from uplift_train"
        ),
    },
    "uplift_sample_submission": {
        "candidate_key": "client_id",
        "validated_checks": (
            "Unique non-null key; exact customer-set match with uplift_test"
        ),
        "main_issues": "Example uplift values are not observed effects",
        "planned_action": "Use only as an optional output-format reference",
    },
}

quality_notes_df = pd.DataFrame.from_dict(
    quality_notes,
    orient="index"
).rename_axis("table_name").reset_index()

data_quality_summary_df = row_counts_df.merge(
    quality_notes_df,
    on="table_name",
    how="left",
    validate="one_to_one"
)

assert data_quality_summary_df["planned_action"].notna().all()

with pd.option_context("display.max_colwidth", None):
    display(data_quality_summary_df)

,table_name,row_count,column_count,candidate_key,validated_checks,main_issues,planned_action
0,clients,400162,5,client_id,Unique non-null key; complete train/test coverage,"1,187 ages flagged; 35,469 missing redemption dates; 536 redemption-before-issue cases; 44,953 post-cutoff redemption dates",Retain customers; preserve raw age; apply provisional age rule; exclude redemption-date features from primary model
1,products,43038,11,product_id,Unique non-null key; binary product flags valid,"Missing hierarchy, segment, brand, vendor and netto values; segment_id inferred as DOUBLE",Retain products; preserve unknown categories; review segment coding and attribute definitions
2,purchases,45786568,13,Order: client_id + transaction_id + store_id + transaction_datetime; detail key unconfirmed,"No identical parsed rows; complete customer/product matches; 8,045,229 candidate orders with consistent purchase_sum","28 transaction IDs have multiple clients; 2,707,278 zero-quantity rows; 93.3532% missing trn_sum_from_red; negative points spent","Use composite order key; retain amount once per order; review quantities, point aggregation and amount definitions"
3,uplift_train,200039,3,client_id,"Unique non-null key; binary fields valid; 100,058 control and 99,981 treatment customers","Assignment mechanism, expected allocation ratio, campaign timing and outcome window require source confirmation",Confirm experiment design; assess SRM where justified and pre-treatment covariate balance before effect reporting
4,uplift_test,200123,1,client_id,Unique non-null key; no train overlap; all IDs present in clients,No public treatment assignment or outcome,Use for optional scoring; evaluate models on held-out customers from uplift_train
5,uplift_sample_submission,200123,2,client_id,Unique non-null key; exact customer-set match with uplift_test,Example uplift values are not observed effects,Use only as an optional output-format reference
